### Instructions
- Set `CSV_PATH` to your desired CSV file.
- Run all cells in order.
- The notebook will print total and per-abstract counts for simplified sentences, text triples, and RDF triples.

In [ ]:
import pandas as pd
import ast
import json

# === CONFIG ===
CSV_PATH = "PATH_to_CSV"
N = 10001 #Number of abstracts to read

In [ ]:

# === Count helper for 'relations' column ===
def count_relations(s):
    if not isinstance(s, str) or s.strip() in ("['None']", "None", "", "[]"):
        return 0
    try:
        s = s.strip()[1:-1]
        if not s:
            return 0
        return s.count("', '") + 1
    except:
        return 0

# === Count helper for 'rdf_triples' using ast.literal_eval
def is_valid_rdf_triple(triple_str):
    try:
        parts = triple_str.strip("\"'").split(" | ")
        # Only check that it looks like a triple (3 parts)
        return len(parts) == 3
    except:
        return False


def count_valid_rdf_triples(cell):
    if not isinstance(cell, str) or cell.strip() in ("None", "", "['None']", "[]"):
        return 0
    try:
        parsed = json.loads(cell)
    except:
        try:
            parsed = ast.literal_eval(cell)
        except:
            return 0

    if not isinstance(parsed, list):
        return 0

    count = 0
    for t in parsed:
        if is_valid_rdf_triple(t):
            count += 1
        else:
            print("❌ Invalid triple:", t)
    return count

In [ ]:
# === Load and trim both CSVs ===
df_full = pd.read_csv(CSV_PATH).iloc[:N].copy()

df_full = df_full.reset_index(drop=True)

# === Sentence counts ===
total_spacy_sentences = df_full['nsent_spacy'].sum()
total_simplified_sentences = df_full['simplified_sentences'].sum()

# === Triple counts ===
df_full['n_text_triples'] = df_full['relations'].apply(count_relations)
df_full['n_rdf_triples'] = df_full['rdf_triples'].apply(count_valid_rdf_triples)

total_text_triples = df_full['n_text_triples'].sum()
total_rdf_triples = df_full['n_rdf_triples'].sum()


total_abstracts = len(df_full)
# === Print Results ===
print("=== Test Results ===")
print(f"Total abstracts:                                {total_abstracts}")
print(f"Simplified sentences:                           {total_simplified_sentences}")
print(f"Text triples:                                   {total_text_triples}")
print(f"Total RDF triples:                              {total_rdf_triples}")

print("=== Test Results Per Abstract ===")
print(f"Simplified sentences per abstract:              {total_simplified_sentences/total_abstracts}")
print(f"Text triples per abstract:                      {total_text_triples/total_abstracts}")
print(f"Total RDF triples per abstract:                 {total_rdf_triples/total_abstracts}")